In [1]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import matthews_corrcoef, confusion_matrix

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample
import matplotlib.pyplot as plt
import pandas as pd
import model as m
import seaborn as sns
import divide_bins as divide
import numpy as np
import feature_selection as fs
import getting_threshold_genes as gtg


2024-09-25 16:37:51.327525: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-09-25 16:37:51.330701: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-09-25 16:37:51.371585: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-09-25 16:37:51.371608: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-09-25 16:37:51.372330: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

In [6]:
file = "RNAs_z_Combat_gene_ranking.csv"
feature_scoring_path = f"/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Results/feature_selection/RNAseq/{file}"


In [7]:
rn_df = gtg.get_gene_rank(feature_scoring_path)
rn_df.to_csv("Results/feature_selection/RNAseq/"+file.split(".")[0]+"_gene_ranking.csv")

/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":


In [8]:
top_df = gtg.get_df(feature_scoring_path)
top_df

 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  5 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
0.14501340549984


/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/.venv/lib/python3.10/site-packages/pandas/core/indexes/base.py:945: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*new_inputs, **kwargs)
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/.venv/lib/python3.10/site-packages/numpy/lib/polynomial.py:668: RuntimeWarning: invalid value encountered in divide
  lhs /= scale


,Ensembl,coef,abs_coef
0,ENSG00000232177.1,-0.815118,0.815118
1,ENSG00000189058.8,0.697045,0.697045
2,ENSG00000249072.1,-0.646121,0.646121
3,ENSG00000127720.7,0.629600,0.629600
4,ENSG00000270388.1,-0.595745,0.595745
...,...,...,...
453,ENSG00000133315.10,-0.145635,0.145635
454,ENSG00000112562.18,0.145291,0.145291
455,ENSG00000146574.15,0.145238,0.145238
456,ENSG00000202430.1,-0.145195,0.145195


In [9]:
top_df.to_csv("Results/feature_selection/RNAseq/"+file.split(".")[0]+"_top_genes_inflexion.csv")

In [10]:
top_df = gtg.get_df(feature_scoring_path, inflexion=False)
top_df

0.14501340549984


/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":


,Ensembl,coef,abs_coef
0,ENSG00000232177.1,-0.815118,0.815118
1,ENSG00000189058.8,0.697045,0.697045
2,ENSG00000249072.1,-0.646121,0.646121
3,ENSG00000127720.7,0.629600,0.629600
4,ENSG00000270388.1,-0.595745,0.595745
...,...,...,...
453,ENSG00000133315.10,-0.145635,0.145635
454,ENSG00000112562.18,0.145291,0.145291
455,ENSG00000146574.15,0.145238,0.145238
456,ENSG00000202430.1,-0.145195,0.145195


In [11]:
top_df.to_csv("Results/feature_selection/RNAseq/"+file.split(".")[0]+"_top_genes_elbow.csv")

In [16]:
files = ["Sex_female_feature_selection_Symbols.csv",
"Sex_male_feature_selection_Symbols.csv",
"Status_Healthy_feature_selection_Symbols.csv",
"Status_Sarcopenia_feature_selection_Symbols.csv",
"Status_trained_feature_selection_Symbols.csv",
"Status_untrained_feature_selection_Symbols.csv"
]

In [33]:
# Dataframe with the name of the file, the treshold and the number of genes
result_data = {}



In [34]:
for file in files:
    print(file)
    feature_scoring_path = f"/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Results/feature_selection/RNAseq/ridge_L2_symbols/{file}"
    rn_df = gtg.get_gene_rank(feature_scoring_path)
    #rn_df.to_csv("Results/feature_selection/RNAseq/"+file.split(".")[0]+"_gene_ranking.csv")
    top_df = gtg.get_df(feature_scoring_path)
    #print(len(top_df))
    result_data[file+"1"]= {"File": file, "Cut": "inflexion", "Threshold": 0, "Number of genes": len(top_df)}
    top_df.to_csv("Results/feature_selection/RNAseq/"+file.split(".")[0]+"_top_genes_inflexion.csv")
    top_df = gtg.get_df(feature_scoring_path, inflexion=False)
    result_data[file+"2"]= {"File": file, "Cut": "elbow", "Threshold": 0, "Number of genes": len(top_df)}
    #top_df.to_csv("Results/feature_selection/RNAseq/"+file.split(".")[0]+"_top_genes_elbow.csv")


Sex_female_feature_selection_Symbols.csv
0.04729346474757676
0.0604903282399061
Sex_male_feature_selection_Symbols.csv


/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent w

0.07947660620005964
0.1313989771298354
Status_Healthy_feature_selection_Symbols.csv
0.11034613570996898


/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent w

0.2125045624629661
Status_Sarcopenia_feature_selection_Symbols.csv
0.0065424904878006
0.0091703289695409
Status_trained_feature_selection_Symbols.csv


/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent w

0.0004354707211745479
0.0005771539515332
Status_untrained_feature_selection_Symbols.csv
0.0003063085332272971
0.0003950293490687


/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":


In [35]:
pd.DataFrame(result_data).T.to_csv("Results/feature_selection/RNAseq/feature_selection_results_alpha_and_number_genes.csv")